# 06 — Build the Stage B mix (Kaggle, CPU)

Downloads and converts the Stage B judge sources (`DATASETS.md` section 8, todo Phase 4b), builds the `stage_b` mix with a replay of Stage A, and lists the judge-benchmark records whose passages the mix trains on (eval v2). **Nothing big leaves Kaggle:** the mix stays as this notebook's output and the training notebook takes it as an input; only the small hand-off files go to the laptop.

- **Settings:** Accelerator → None (CPU, no GPU quota), Internet → On.
- **Inputs:** Add Input → the `jevlite-mix` dataset (the Stage A mix: `stage_a/train.e*.jsonl`, test files).
- **Secrets (optional):** `HF_TOKEN` with access to `lytang/LLM-AggreFact`. Without it MiniCheck C2D is left out (its claims must be checked against LLM-AggreFact first); D2C is kept.
- **Disk:** downloads in `/tmp/raw`, converted sources in `/tmp/public` (not saved); output = `stage_b/` (the mix) + `handoff/` (KB).
- **Smoke run first:** run all cells with `LIMIT = 200`, read the checks and `mix_report.md`. Then `LIMIT = None` and **Save Version → Save & Run All (Commit)**.


In [ ]:
# Environment: code from GitHub at a fixed commit, kept outside /kaggle/working so it is not saved as output.
COMMIT = "main"   # set the commit hash of the pushed Stage B code for a reproducible run

!rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
!cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
!pip install -q loguru
!python --version
!df -h /kaggle/working /tmp | tail -2

In [ ]:
# Parameters
LIMIT = 200          # smoke run; None = full conversion
STAGE_B = ["ragbench_train", "hotpotqa_judge", "tatqa_judge", "finqa_judge", "clapnq_judge",
           "faithfulness_qa", "minicheck", "trust_data", "psiloqa", "qasper", "ragtruth",
           "helpsteer2", "helpsteer3"]   # Score data of Stage A, kept directly (the replay holds few Score records)
JUDGE_BENCH = ["contextual_judge_bench", "ragbench", "halubench", "faithbench"]   # eval only: for the overlap ids
RAW, PUBLIC = "/tmp/raw", "/tmp/public"
OUT = "/kaggle/working/stage_b"
HANDOFF = "/kaggle/working/handoff"
EPOCHS, BATCH = 2, 8   # BATCH = train.per_device_batch_size of Stage B at 1024 tokens (for the loss weights)

import glob, os
os.makedirs(HANDOFF, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "- MiniCheck C2D will be left out")

hits = glob.glob("/kaggle/input/**/stage_a/train.e0.jsonl", recursive=True)
assert hits, "add the jevlite-mix dataset as an input"
os.environ["STAGE_A_MIX"] = os.path.dirname(hits[0])
print("STAGE_A_MIX =", os.environ["STAGE_A_MIX"]); print(sorted(os.listdir(os.environ["STAGE_A_MIX"])))

## Convert

One dataset per call, so a failure in one does not stop the others. The judge benchmarks are converted too (eval only, ~70 MB), only to find the eval records whose passages the mix trains on.

In [ ]:
import subprocess, sys

for name in STAGE_B + JUDGE_BENCH:
    cmd = [sys.executable, "/tmp/JEV_Judge/scripts/convert_public.py", "--datasets", name, "--out", PUBLIC, "--raw-dir", RAW]
    if LIMIT is not None and name in STAGE_B:   # the eval sets are always converted whole
        cmd += ["--limit", str(LIMIT)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=False, cwd="/tmp/JEV_Judge")
!du -sh {RAW} {PUBLIC}; df -h /tmp | tail -1

## Check

Per source: records by split, criterion and label; label source; three random records. Then a data-level look at the passage shortcut (`DATASETS.md` 5.1): for the answerability records, the share of chunks that occur under **both** labels. ClapNQ-style construction gives 100%; a low share means the label could be read off the chunks alone (the model-level 2×2 swap test runs after Stage B).

In [ ]:
import json, random, hashlib
from collections import Counter, defaultdict
from pathlib import Path

stats = {}
for name in STAGE_B:
    path = Path(PUBLIC) / f"{name}.jsonl"
    if not path.exists():
        print(f"== {name}: MISSING"); stats[name] = None; continue
    recs = [json.loads(l) for l in path.open()]
    by = Counter((r["split"], r.get("criterion"), r["type"], str(r["target"]) if r["type"] != "bool" else int(r["target"])) for r in recs)
    chunks = defaultdict(set)
    for r in recs:
        if r.get("criterion") == "answerable":
            for c in (r["state"] if isinstance(r["state"], list) else [r["state"]]):
                chunks[hashlib.md5(c.encode()).hexdigest()].add(int(r["target"]))
    both = sum(len(v) == 2 for v in chunks.values()) / len(chunks) if chunks else None
    stats[name] = {"n": len(recs), "by_split_criterion_label": {" ".join(map(str, k)): v for k, v in sorted(by.items())},
                   "label_source": dict(Counter(r.get("label_source") for r in recs)),
                   "perturbation": dict(Counter(r.get("perturbation") for r in recs if r.get("perturbation"))),
                   "answerable_chunks_with_both_labels": both}
    print(f"== {name}: {len(recs)} records; answerable chunks with both labels: {both}")
    for k, v in by.most_common(8):
        print("   ", k, v)
    for r in random.Random(0).sample(recs, min(3, len(recs))):
        state = r["state"] if isinstance(r["state"], str) else " ||| ".join(map(str, r["state"].values() if isinstance(r["state"], dict) else r["state"]))
        print(f"  [{r['type']} → {r['target']}] {r['prompt'][:200]}")
        print("   ", state[:300].replace("\n", " / "))
    print()
Path(HANDOFF, "source_stats.json").write_text(json.dumps(stats, indent=2))

## Build the mix

`configs/mixes/stage_b.yaml`: the Stage B sources + a replay of Stage A (`$STAGE_A_MIX/train.e*.jsonl`) + the Stage A test sets unchanged. `fit_only` sources are tokenized to drop records longer than 1024 tokens (several minutes). With `run_repeat`, `--batch-size` adds the suggested `loss.weights` to the report.

In [ ]:
# smoke run: a few hundred records per source cannot fill 150k per epoch, so build a small copy of the mix
config = "/tmp/JEV_Judge/configs/mixes/stage_b.yaml"
if LIMIT is not None:
    Path("/tmp/stage_b_smoke.yaml").write_text(f"extends: {config}\ntotal: 5000\nmax_repeat: 50\n")
    config = "/tmp/stage_b_smoke.yaml"
cmd = [sys.executable, "/tmp/JEV_Judge/scripts/build_mix.py", "--config", config,
       "--dir", PUBLIC, "--out", OUT, "--epochs", str(EPOCHS), "--batch-size", str(BATCH)]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd="/tmp/JEV_Judge")
!cp {OUT}/mix_report.md {OUT}/mix_report.json {HANDOFF}/
!du -sh {OUT}; ls -la {OUT}
print(open(f"{OUT}/mix_report.md").read())

## eval v2: overlap of the mix with the judge benchmarks

Record ids of the judge benchmarks whose passages occur in the Stage B training files (`prefix` mode, as for Stage A). On the laptop they are added to eval v2 with `make_eval_v2.py --exclude`.

In [ ]:
train_files = sorted(glob.glob(f"{OUT}/train.e*.jsonl"))
cmd = [sys.executable, "/tmp/JEV_Judge/scripts/overlap_ids.py", "--train", *train_files,
       "--data", *[f"{PUBLIC}/{n}.jsonl" for n in JUDGE_BENCH], "--out", f"{HANDOFF}/stage_b_overlap_ids.txt"]
subprocess.run(cmd, check=True, cwd="/tmp/JEV_Judge")
!ls -la {HANDOFF}

## Hand-off

- **Training:** in the Stage B training notebook, Add Input → Notebook Output → this notebook's version; the mix is under `stage_b/`.
- **To the laptop (KB only):** from this version's Output tab download the files of `handoff/` (do **not** use `kaggle kernels output`: it downloads the whole mix). Put them in `data/judge_bench/stage_b_handoff/`, then:

```bash
uv run python scripts/make_eval_v2.py --data data/public/{contextual_judge_bench,ragbench,halubench,faithbench}.jsonl \
  --exclude docs/judge_bench/overlap_ids.txt --exclude data/judge_bench/stage_b_handoff/stage_b_overlap_ids.txt \
  --out docs/judge_bench/eval_v2
```

and re-score Stage A on eval v2 (`JUDGE_BENCH.md` section 9).